# Level 3 Task 1: Predictive Modeling (Classification)

This notebook preprocesses the churn data, tunes multiple classification models, and compares them using accuracy, precision, recall, and F1-score.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
base_dir = Path.cwd()
if not (base_dir / "data" / "churn_train.csv").exists():
    base_dir = Path("Level_3_Advanced") / "Task_1_Predictive_Modeling"

train_path = base_dir / "data" / "churn_train.csv"
test_path = base_dir / "data" / "churn_test.csv"
output_dir = base_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
target_column = "Churn"
sns.set_theme(style="whitegrid")

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

## Preprocess Features

In [ ]:
y_train_full = train_df[target_column].astype(int).to_numpy()
y_test = test_df[target_column].astype(int).to_numpy()
x_train_df = train_df.drop(columns=[target_column]).copy()
x_test_df = test_df.drop(columns=[target_column]).copy()
x_train_df["Area code"] = x_train_df["Area code"].astype(str)
x_test_df["Area code"] = x_test_df["Area code"].astype(str)
combined = pd.concat([x_train_df, x_test_df], axis=0, ignore_index=True)
combined_encoded = pd.get_dummies(combined, columns=["State", "Area code", "International plan", "Voice mail plan"], drop_first=False)
x_train_full_df = combined_encoded.iloc[: len(train_df)].copy()
x_test_full_df = combined_encoded.iloc[len(train_df):].copy()
x_train_full_df.shape, x_test_full_df.shape

## Validation Split

In [ ]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(x_train_full_df))
split_index = int(len(x_train_full_df) * 0.8)
train_idx, val_idx = indices[:split_index], indices[split_index:]
x_train_df = x_train_full_df.iloc[train_idx]
x_val_df = x_train_full_df.iloc[val_idx]
y_train = y_train_full[train_idx]
y_val = y_train_full[val_idx]
means = x_train_df.mean()
stds = x_train_df.std(ddof=0).replace(0, 1)
x_train = ((x_train_df - means) / stds).to_numpy(dtype=float)
x_val = ((x_val_df - means) / stds).to_numpy(dtype=float)
x_test = ((x_test_full_df - means) / stds).to_numpy(dtype=float)
x_train.shape, x_val.shape, x_test.shape

## Model Helpers

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def fit_logistic_regression(x, y, learning_rate, epochs, l2_penalty):
    weights = np.zeros(x.shape[1] + 1)
    x_bias = np.column_stack([np.ones(len(x)), x])
    for _ in range(epochs):
        preds = sigmoid(x_bias @ weights)
        errors = preds - y
        grad = (x_bias.T @ errors) / len(x)
        grad[1:] += l2_penalty * weights[1:] / len(x)
        weights -= learning_rate * grad
    return weights

def predict_logistic(weights, x, threshold=0.5):
    x_bias = np.column_stack([np.ones(len(x)), x])
    probs = sigmoid(x_bias @ weights)
    labels = (probs >= threshold).astype(int)
    return probs, labels

def fit_gaussian_nb(x, y):
    classes = np.unique(y)
    means = {}
    variances = {}
    priors = {}
    for c in classes:
        x_c = x[y == c]
        means[c] = x_c.mean(axis=0)
        variances[c] = x_c.var(axis=0) + 1e-6
        priors[c] = len(x_c) / len(x)
    return {"classes": classes, "means": means, "variances": variances, "priors": priors}

def predict_gaussian_nb(model, x):
    log_posteriors = []
    for c in model["classes"]:
        mean = model["means"][c]
        variance = model["variances"][c]
        prior = np.log(model["priors"][c])
        log_likelihood = -0.5 * np.sum(np.log(2 * np.pi * variance) + ((x - mean) ** 2) / variance, axis=1)
        log_posteriors.append(prior + log_likelihood)
    log_posteriors = np.column_stack(log_posteriors)
    max_log = log_posteriors.max(axis=1, keepdims=True)
    exp_scores = np.exp(log_posteriors - max_log)
    probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    labels = model["classes"][probs.argmax(axis=1)].astype(int)
    positive_index = list(model["classes"]).index(1)
    return probs[:, positive_index], labels

def predict_knn(x_train, y_train, x_eval, k):
    distances = np.sqrt(((x_eval[:, None, :] - x_train[None, :, :]) ** 2).sum(axis=2))
    nearest_indices = np.argsort(distances, axis=1)[:, :k]
    nearest_labels = y_train[nearest_indices]
    probs = nearest_labels.mean(axis=1)
    labels = (probs >= 0.5).astype(int)
    return probs, labels

def classification_metrics(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
    return {"accuracy": round(float(accuracy), 4), "precision": round(float(precision), 4), "recall": round(float(recall), 4), "f1_score": round(float(f1), 4), "tp": tp, "tn": tn, "fp": fp, "fn": fn}

## Tune Models on Validation Data

In [ ]:
logistic_grid = [
    {"learning_rate": 0.05, "epochs": 1500, "l2_penalty": 0.0},
    {"learning_rate": 0.05, "epochs": 2000, "l2_penalty": 0.01},
    {"learning_rate": 0.1, "epochs": 1500, "l2_penalty": 0.01},
    {"learning_rate": 0.1, "epochs": 2000, "l2_penalty": 0.1},
]
best_logistic = None
best_logistic_f1 = -1.0
for params in logistic_grid:
    weights = fit_logistic_regression(x_train, y_train, **params)
    _, val_pred = predict_logistic(weights, x_val)
    metrics = classification_metrics(y_val, val_pred)
    if metrics["f1_score"] > best_logistic_f1:
        best_logistic_f1 = metrics["f1_score"]
        best_logistic = {"weights": weights, "params": params, "metrics": metrics}

best_knn = None
best_knn_f1 = -1.0
for k in [3, 5, 7, 9, 11]:
    _, val_pred = predict_knn(x_train, y_train, x_val, k=k)
    metrics = classification_metrics(y_val, val_pred)
    if metrics["f1_score"] > best_knn_f1:
        best_knn_f1 = metrics["f1_score"]
        best_knn = {"k": k, "metrics": metrics}

nb_model = fit_gaussian_nb(x_train, y_train)
_, nb_val_pred = predict_gaussian_nb(nb_model, x_val)
nb_metrics = classification_metrics(y_val, nb_val_pred)

validation_df = pd.DataFrame([
    {"model": "Logistic Regression", "hyperparameters": str(best_logistic["params"]), **best_logistic["metrics"]},
    {"model": "KNN", "hyperparameters": f"{{'k': {best_knn['k']}}}", **best_knn["metrics"]},
    {"model": "Gaussian Naive Bayes", "hyperparameters": "{}", **nb_metrics},
]).sort_values(by=["f1_score", "recall", "precision"], ascending=False).reset_index(drop=True)
validation_df

## Retrain and Evaluate on Test Data

In [ ]:
means_all = x_train_full_df.mean()
stds_all = x_train_full_df.std(ddof=0).replace(0, 1)
x_train_all = ((x_train_full_df - means_all) / stds_all).to_numpy(dtype=float)
x_test_scaled = ((x_test_full_df - means_all) / stds_all).to_numpy(dtype=float)
results = []
for _, row in validation_df.iterrows():
    if row['model'] == 'Logistic Regression':
        params = best_logistic['params']
        final_weights = fit_logistic_regression(x_train_all, y_train_full, **params)
        _, pred = predict_logistic(final_weights, x_test_scaled)
        hyper = str(params)
    elif row['model'] == 'KNN':
        _, pred = predict_knn(x_train_all, y_train_full, x_test_scaled, k=best_knn['k'])
        hyper = f"{{'k': {best_knn['k']}}}"
    else:
        final_nb = fit_gaussian_nb(x_train_all, y_train_full)
        _, pred = predict_gaussian_nb(final_nb, x_test_scaled)
        hyper = '{}'
    metrics = classification_metrics(y_test, pred)
    results.append({"model": row['model'], "hyperparameters": hyper, **metrics})
results_df = pd.DataFrame(results).sort_values(by=["f1_score", "recall", "precision"], ascending=False).reset_index(drop=True)
results_df

## Compare Models

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=results_df, x='model', y='f1_score', hue='model', palette='Set2', legend=False)
plt.title('Model Comparison by F1-score', fontweight='bold')
plt.xlabel('Model')
plt.ylabel('F1-score')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()